# 04 Deep Learning Models for ADR Prediction

This notebook trains and evaluates deep learning models (MLP, ResNet, Attention) on the processed ADR dataset.

In [ ]:
# Core imports for data handling, modeling, and evaluation
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Resolve project paths and import shared training/model code from src/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'processed_data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from models import get_model
from train import ADRDataset, Trainer

DATA_DIR = PROJECT_ROOT / 'processed_data'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load train/validation/test splits produced by src/preprocessing.py
X_train = pd.read_csv(DATA_DIR / 'X_train.csv')
y_train = pd.read_csv(DATA_DIR / 'y_train.csv')
X_val = pd.read_csv(DATA_DIR / 'X_val.csv')
y_val = pd.read_csv(DATA_DIR / 'y_val.csv')
X_test = pd.read_csv(DATA_DIR / 'X_test.csv')
y_test = pd.read_csv(DATA_DIR / 'y_test.csv')

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'ADR rate - Train: {y_train.mean().values[0]:.3f}, Val: {y_val.mean().values[0]:.3f}, Test: {y_test.mean().values[0]:.3f}')

In [ ]:
# Build DataLoaders (num_workers=0 for notebook portability)
BATCH_SIZE = 1024
train_dataset = ADRDataset(X_train, y_train)
val_dataset = ADRDataset(X_val, y_val)
test_dataset = ADRDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# Deep model training configuration
LEARNING_RATE = 1e-3
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10
ENABLE_FINE_TUNING = True
FINE_TUNE_LR_FACTOR = 0.1
FINE_TUNE_EPOCHS = 15
FINE_TUNE_PATIENCE = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

models_config = {
    'mlp': {
        'model_type': 'mlp',
        'input_dim': X_train.shape[1],
        'hidden_dims': [256, 128, 64, 32],
        'dropout_rate': 0.3
    },
    'resnet': {
        'model_type': 'resnet',
        'input_dim': X_train.shape[1],
        'hidden_dim': 256,
        'num_blocks': 3,
        'dropout_rate': 0.3
    },
    'attention': {
        'model_type': 'attention',
        'input_dim': X_train.shape[1],
        'hidden_dims': [256, 128],
        'dropout_rate': 0.3
    }
}

training_results = {}
prediction_results = {}

for model_name, cfg in models_config.items():
    print(f'\n=== Training {model_name.upper()} ===')
    cfg_local = cfg.copy()
    model_type = cfg_local.pop('model_type')
    model = get_model(model_type, **cfg_local)

    trainer = Trainer(model, device=DEVICE, learning_rate=LEARNING_RATE, loss_type='focal', class_weights=None)

    history = trainer.fit(
        train_loader,
        val_loader,
        epochs=EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )

    if ENABLE_FINE_TUNING:
        fine_tune_lr = LEARNING_RATE * FINE_TUNE_LR_FACTOR
        print(f'Fine-tuning {model_name.upper()} at lr={fine_tune_lr}')
        trainer.set_learning_rate(fine_tune_lr)
        history = trainer.fit(
            train_loader,
            val_loader,
            epochs=FINE_TUNE_EPOCHS,
            early_stopping_patience=FINE_TUNE_PATIENCE
        )

    # Save model checkpoint + history
    trainer.save_model(MODELS_DIR / f'{model_name}_best.pth')
    with open(MODELS_DIR / f'{model_name}_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # Predict on validation and test for metrics/threshold tuning
    val_proba, val_true = trainer.predict(val_loader)
    test_proba, test_true = trainer.predict(test_loader)

    val_proba = val_proba.flatten()
    val_true = val_true.flatten()
    test_proba = test_proba.flatten()
    test_true = test_true.flatten()

    threshold_grid = np.linspace(0.05, 0.95, 19)
    val_f1_scores = [f1_score(val_true, (val_proba >= t).astype(int)) for t in threshold_grid]
    best_threshold = float(threshold_grid[int(np.argmax(val_f1_scores))])
    test_pred = (test_proba >= best_threshold).astype(int)

    training_results[model_name] = {
        'val_auroc': float(roc_auc_score(val_true, val_proba)),
        'val_auprc': float(average_precision_score(val_true, val_proba)),
        'test_auroc': float(roc_auc_score(test_true, test_proba)),
        'test_auprc': float(average_precision_score(test_true, test_proba)),
        'selected_threshold': best_threshold
    }

    prediction_results[model_name] = {
        'y_true': test_true.tolist(),
        'y_proba': test_proba.tolist(),
        'y_pred': test_pred.tolist()
    }

results_df = pd.DataFrame({
    'Model': list(training_results.keys()),
    'Val AUROC': [v['val_auroc'] for v in training_results.values()],
    'Val AUPRC': [v['val_auprc'] for v in training_results.values()],
    'Test AUROC': [v['test_auroc'] for v in training_results.values()],
    'Test AUPRC': [v['test_auprc'] for v in training_results.values()],
    'Selected Threshold': [v['selected_threshold'] for v in training_results.values()]
})

display(results_df.sort_values('Val AUROC', ascending=False))

In [ ]:
# Save notebook-level deep learning artifacts
results_df.to_csv(RESULTS_DIR / 'deep_models_metrics.csv', index=False)

with open(RESULTS_DIR / 'deep_models_metrics.json', 'w') as f:
    json.dump(training_results, f, indent=2)

with open(RESULTS_DIR / 'deep_models_test_predictions.json', 'w') as f:
    json.dump(prediction_results, f, indent=2)

# Print a detailed report for the best validation model
best_model_name = results_df.sort_values('Val AUROC', ascending=False).iloc[0]['Model']
best = prediction_results[best_model_name]
print(f'Best model: {best_model_name}')
print(classification_report(best['y_true'], best['y_pred'], target_names=['No ADR', 'ADR']))

cm = confusion_matrix(best['y_true'], best['y_pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No ADR', 'ADR'], yticklabels=['No ADR', 'ADR'])
plt.title(f'Confusion Matrix - {best_model_name.upper()}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'deep_best_model_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()